# Example 11.5 / 1: A minimum-norm blend

Companion notebook for **11.5 Second-Order Cones** in the AMPL lab Documentation, section "A minimum-norm blend". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A minimum-norm blend

This model finds the smallest Euclidean-norm nonnegative blend that achieves a required score. The objective variable `Radius` and the cone constraint expose the geometry directly.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS;
param score {ITEMS} > 0;
param required > 0;

var Mix {ITEMS} >= 0;
var Radius >= 0;

minimize MinRadius: Radius;

subject to Requirement:
    sum {i in ITEMS} score[i] * Mix[i] >= required;

subject to NormCone:
    sqrt(sum {i in ITEMS} Mix[i]^2) <= Radius;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ITEMS"] = ["A", "B"]
ampl.param["score"] = {"A": 3, "B": 4}
ampl.param["required"] = 10

In [5]:
# run.py
ampl.solve(solver="gurobi", verbose=False)
print(ampl.solve_result)
print(ampl.var["Mix"].to_dict())
print(round(ampl.var["Radius"].value(), 6))

solved
{'A': 1.200142559417847, 'B': 1.599893095921253}
2.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set ITEMS := A B;
param score := A 3 B 4;
param required := 10;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver gurobi;
option solver_msg 0;
solve;
display Mix, Radius;

Mix [*] :=
A  1.20014
B  1.59989
;

Radius = 2

